<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 0. Settings
Band power for all 60 final sessions with the new trial filter (trials_filtered.csv, keep == True; all long_prob blocks)
and the new bad-channel list (extra_bad_channels.csv, removed before referencing).
Two references: CMR (global median) and LOCAL (median of channels 30–200 µm away).
Output: one file per session and reference in data\analysis_main\band_power_v3 (CMR) and band_power_v3_localref (LOCAL).
The old files (band_power_v1, v2, v2_localref) are not touched.

<div>

In [7]:
# %%
# ── 0. SETTINGS ──
import os, sys, time, glob, pickle, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="The following channels did not have enough neighbors")
MAIN      = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"
sys.path.insert(0, MAIN)
import functions as fn
importlib.reload(fn)

DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"
PROC      = os.path.join(DATA_ROOT, "2_processed")
AMIR      = os.path.join(DATA_ROOT, "amir_data")
TABLES    = os.path.join(MAIN, "tables")
FIGS      = os.path.join(MAIN, "figures", "06_band_power")
BP_DIRS   = {"CMR":   os.path.join(DATA_ROOT, "analysis_main", "band_power_v3"),
             "LOCAL": os.path.join(DATA_ROOT, "analysis_main", "band_power_v3_localref")}
for d in [FIGS, *BP_DIRS.values()]:
    os.makedirs(d, exist_ok=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

final = pd.read_csv(os.path.join(TABLES, "final_sessions.csv"), dtype={"rat": str})
XB    = pd.read_csv(os.path.join(TABLES, "extra_bad_channels.csv"))
TR    = pd.read_csv(os.path.join(TABLES, "trials_filtered.csv"), dtype={"rat": str}, low_memory=False)
K     = TR[TR["keep"]].copy()
TEST  = "20260529_r0000136_wt_1318"

print(f"final sessions: {len(final)} | kept trials: {len(K)} | extra bad channel rows: {len(XB)} (expected 107)")
print("kept trials with a tone, by label:", K.groupby("label")["tone_neural_s"].apply(lambda v: v.notna().sum()).to_dict())
print("output folders:", {r: os.path.basename(d) for r, d in BP_DIRS.items()})

BURST_DIR = os.path.join(DATA_ROOT, "analysis_main", "bursts_karvat2021_v2")   # beta + gamma, CMR + LOCAL
os.makedirs(BURST_DIR, exist_ok=True)
REP   = pd.read_csv(os.path.join(DATA_ROOT, "analysis_main", "band_power_v1", "_representative_channels.csv"))
CHREG = pd.read_csv(os.path.join(TABLES, "channel_regions_all.csv"))
CHREG["channel_id"] = CHREG["channel_id"].astype(str)
warnings.filterwarnings("ignore", message="The following channels did not have enough neighbors")
print("representative channels:", REP[["rat", "region", "x", "y"]].to_dict("records"))

final sessions: 60 | kept trials: 8615 | extra bad channel rows: 107 (expected 107)
kept trials with a tone, by label: {'correct': 4486, 'early': 0, 'late': 1549}
output folders: {'CMR': 'band_power_v3', 'LOCAL': 'band_power_v3_localref'}
representative channels: [{'rat': 134, 'region': 'Cg1', 'x': 750, 'y': 3105}, {'rat': 134, 'region': 'IL', 'x': 250, 'y': 375}, {'rat': 134, 'region': 'PL', 'x': 532, 'y': 1845}, {'rat': 135, 'region': 'Cg1', 'x': 250, 'y': 4020}, {'rat': 135, 'region': 'IL', 'x': 0, 'y': 1065}, {'rat': 135, 'region': 'PL', 'x': 282, 'y': 2145}, {'rat': 136, 'region': 'Cg1', 'x': 32, 'y': 3255}, {'rat': 136, 'region': 'IL', 'x': 250, 'y': 585}, {'rat': 136, 'region': 'PL', 'x': 250, 'y': 2430}]


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 1. Loading with CMR and LOCAL reference (test)
fn.load_session(..., refs=("CMR", "LOCAL")): extra bad channels removed first, then CMR (global median) and LOCAL
(median of the channels 30–200 µm away, SpikeInterface; same as band_power_v2_localref, 2026-10-20).
Test: channel count and signal size (RMS of 10 s, median over channels) in both references.

<div>

In [5]:
# %%
# ── 1. LOAD WITH CMR + LOCAL (test session) ──
importlib.reload(fn)
recs, removed = fn.load_session(PROC, TEST, XB, refs=("original", "CMR", "LOCAL"))
fs = recs["CMR"].get_sampling_frequency(); mid = recs["CMR"].get_num_samples() // 2
for ref, rec in recs.items():
    x = rec.get_traces(start_frame=mid, end_frame=mid + int(10 * fs), return_in_uV=True)
    print(f"{ref:8s}: {rec.get_num_channels()} channels | RMS (10 s, median over channels) {np.median(x.std(axis=0)):.1f} µV")
print("extra bad removed:", removed)

original: 383 channels | RMS (10 s, median over channels) 72.7 µV
CMR     : 383 channels | RMS (10 s, median over channels) 44.8 µV
LOCAL   : 383 channels | RMS (10 s, median over channels) 22.2 µV
extra bad removed: []


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 2. Test band power on one session (CMR)
fn.band_power_session on 20260529_r0000136_wt_1318, CMR, all kept trials (all long_prob blocks).
Conditions for the time courses and spectra: correct / early / late × long / short, plus "early long (hold>0.3)"
(early long trials held > 0.3 s, as in the old files) -> both early-long versions are saved; window values are saved per trial,
so any trial selection is possible later for them.
Sanity check: theta 4–7 hold vs pre, correct long, median over channels (2026-10-19 test: +30.7 %).
The file is saved in band_power_v3 (the loop in step 3 then skips this session).

<div>

In [6]:
# %%
# ── 3. TEST BAND POWER ON ONE SESSION (CMR) ──
importlib.reload(fn)
recs, removed = fn.load_session(PROC, TEST, XB, refs=("CMR",))
trs = fn.session_trials(K, TEST)
print(f"{TEST}: {recs['CMR'].get_num_channels()} channels | {len(trs)} kept trials")

t0 = time.time()
r = fn.band_power_session(recs["CMR"], trs, os.path.join(BP_DIRS["CMR"], TEST + ".pkl"),
                          meta_extra={"session": TEST, "ref": "CMR", "extra_bad_removed": removed})
print(f"done in {(time.time() - t0) / 60:.1f} min | trials used {r['meta']['n_trials']}")

T, lp = r["trials"], r["logpow"]
cl = ((T["label"] == "correct") & (T["type"] == "long")).values
print(f"\ncorrect long trials: {cl.sum()}")
for b in fn.MAIN_BANDS:
    d = (lp[b]["hold"][cl] - lp[b]["pre"][cl]).mean(axis=0)                  # per channel, mean over trials
    print(f"  {b:13s} hold vs pre: median over channels {100 * (10 ** np.median(d) - 1):+.1f} %")
print("\ncurve conditions:", list(r["curves"]["Press"]["theta 4-7"]))
print("trials per condition (spectra, pre window):", r["spec_n"]["pre"])

20260529_r0000136_wt_1318: 383 channels | 139 kept trials
    channels 0-64 of 383 done (59 s)
    channels 65-129 of 383 done (93 s)
    channels 130-194 of 383 done (127 s)
    channels 195-259 of 383 done (162 s)
    channels 260-324 of 383 done (195 s)
    channels 325-382 of 383 done (226 s)
done in 3.8 min | trials used 139

correct long trials: 49
  delta 1-3     hold vs pre: median over channels +5.1 %
  theta 4-7     hold vs pre: median over channels +30.7 %
  alpha 8-12    hold vs pre: median over channels +18.5 %
  beta 13-29    hold vs pre: median over channels +4.3 %
  lowG 30-60    hold vs pre: median over channels +3.9 %
  highG 61-100  hold vs pre: median over channels -3.7 %

curve conditions: ['correct long', 'correct short', 'early long', 'early short', 'late long', 'late short', 'early long (hold>0.3)']
trials per condition (spectra, pre window): {'correct long': 49, 'correct short': 48, 'early long': 15, 'early short': 2, 'late long': 3, 'late short': 22, 'early lo

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 3. Bursts: test on one session (CMR and LOCAL)
fn.bursts_session: Karvat et al. 2021 rule (as cell 2.6 of analysis_main), one representative channel per region (PL, IL, Cg1; 1.13b rule).
Beta 15–30 Hz exactly as before (7 cycles); gamma 30–100 Hz added (10 cycles, 48–52 Hz set to 0 because of the notch).
Theta and NeuroDSP dropped. Output: data\analysis_main\bursts_karvat2021_v2\<session>.npz (+ _trials.csv).
Note: gamma is a wide band (~70 frequency rows) -> its occupancy is much higher than beta's; compare conditions within a band.

<div>

In [8]:
# %%
# ── 3. BURSTS: TEST ON ONE SESSION (CMR + LOCAL) ──
importlib.reload(fn)
recs, removed = fn.load_session(PROC, TEST, XB, refs=("CMR", "LOCAL"))
rat = final.set_index("session").loc[TEST, "rat"]
t0 = time.time()
z = fn.bursts_session(recs, fn.session_trials(K, TEST), REP, CHREG, TEST, rat, os.path.join(BURST_DIR, TEST + ".npz"))
print(f"\ndone in {(time.time() - t0) / 60:.1f} min")
print("channels:", {k: (str(v[0]), float(z[k.replace('channel', 'distance_um')][0])) for k, v in z.items() if k.endswith("|channel")})

    CMR Cg1 beta 15-30: occupancy 0.253 (26 s)
    CMR Cg1 gamma 30-100: occupancy 0.503 (39 s)
    CMR IL beta 15-30: occupancy 0.268 (42 s)
    CMR IL gamma 30-100: occupancy 0.502 (55 s)
    CMR PL beta 15-30: occupancy 0.269 (58 s)
    CMR PL gamma 30-100: occupancy 0.512 (71 s)
    LOCAL Cg1 beta 15-30: occupancy 0.251 (77 s)
    LOCAL Cg1 gamma 30-100: occupancy 0.490 (90 s)
    LOCAL IL beta 15-30: occupancy 0.274 (93 s)
    LOCAL IL gamma 30-100: occupancy 0.511 (106 s)
    LOCAL PL beta 15-30: occupancy 0.270 (109 s)
    LOCAL PL gamma 30-100: occupancy 0.500 (121 s)

done in 2.0 min
channels: {'CMR|Cg1|channel': ('imec0.ap#AP99', 0.0), 'CMR|IL|channel': ('imec0.ap#AP174', 0.0), 'CMR|PL|channel': ('imec0.ap#AP36', 0.0), 'LOCAL|Cg1|channel': ('imec0.ap#AP99', 0.0), 'LOCAL|IL|channel': ('imec0.ap#AP174', 0.0), 'LOCAL|PL|channel': ('imec0.ap#AP36', 0.0)}


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 4. Bursts: all final sessions (loop, CMR + LOCAL)
Same as step 3 for every final session; finished sessions are skipped; errors -> tables\bursts_errors.csv.

<div>

In [ ]:
# %%
# ── 4. BURSTS: ALL FINAL SESSIONS ──
importlib.reload(fn)
RUN_BURSTS = True
todo = [s for s in final["session"] if not os.path.exists(os.path.join(BURST_DIR, s + ".npz"))]
print(f"bursts: {len(final) - len(todo)} done, {len(todo)} to do", flush=True)
if RUN_BURSTS:
    t_all = time.time()
    for i, s in enumerate(todo, 1):
        t0 = time.time()
        try:
            recs, _ = fn.load_session(PROC, s, XB, refs=("CMR", "LOCAL"))
            rat = final.set_index("session").loc[s, "rat"]
            fn.bursts_session(recs, fn.session_trials(K, s), REP, CHREG, s, rat, os.path.join(BURST_DIR, s + ".npz"), verbose=False)
            print(f"[{i}/{len(todo)}] {s}: ok ({(time.time() - t0) / 60:.1f} min)", flush=True)
        except Exception as e:
            print(f"[{i}/{len(todo)}] {s}: ERROR {e!r}", flush=True)
            pd.DataFrame([{"session": s, "error": repr(e)}]).to_csv(os.path.join(TABLES, "bursts_errors.csv"), mode="a",
                         header=not os.path.exists(os.path.join(TABLES, "bursts_errors.csv")), index=False)
    print(f"bursts finished in {(time.time() - t_all) / 60:.0f} min", flush=True)

bursts: 1 done, 59 to do
[1/59] 20260510_r0000135_wt_1625: ok (2.0 min)
[2/59] 20260510_r0000136_wt_1926: ok (1.7 min)
[3/59] 20260511_r0000136_wt_0909: ok (0.4 min)
[4/59] 20260511_r0000136_wt_0930: ok (1.1 min)
[5/59] 20260511_r0000136_wt_1832: ok (1.4 min)
[6/59] 20260512_r0000136_wt_1054: ok (1.8 min)
[7/59] 20260513_r0000136_wt_1729: ok (0.6 min)
[8/59] 20260514_r0000136_wt_1317: ok (1.5 min)
[9/59] 20260515_r0000135_wt_1953: ok (1.7 min)
[10/59] 20260515_r0000135_wt_1957: ok (0.7 min)
[11/59] 20260515_r0000136_wt_1817: ok (2.4 min)
[12/59] 20260517_r0000135_wt_1846: ok (3.1 min)
[13/59] 20260517_r0000136_wt_1557: ok (3.4 min)
[14/59] 20260519_r0000136_wt_2031: ok (3.3 min)
[15/59] 20260520_r0000135_wt_1357: ok (2.2 min)
[16/59] 20260520_r0000136_wt_1443: ok (2.2 min)
[17/59] 20260521_r0000135_wt_1309: ok (2.6 min)
[18/59] 20260521_r0000136_wt_1221: ok (2.6 min)
[19/59] 20260522_r0000135_wt_1716: ok (3.1 min)


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 5. Band power: all final sessions (loop, CMR and LOCAL)
fn.band_power_session for every final session and both references (one file each in band_power_v3 / band_power_v3_localref).
Finished files are skipped (the test session from step 2 for CMR); errors -> tables\band_power_errors.csv.
Estimated ~3.8 min per session for CMR, ~5.6 min for LOCAL (about 9.5 h in total).

<div>

In [ ]:
# %%
# ── 5. BAND POWER: ALL FINAL SESSIONS, CMR + LOCAL ──
importlib.reload(fn)
RUN_BP = True
jobs = [(s, ref) for ref in BP_DIRS for s in final["session"]
        if not os.path.exists(os.path.join(BP_DIRS[ref], s + ".pkl"))]
print(f"band power: {2 * len(final) - len(jobs)} done, {len(jobs)} to do", flush=True)
if RUN_BP:
    t_all = time.time()
    for i, (s, ref) in enumerate(jobs, 1):
        t0 = time.time()
        try:
            recs, removed = fn.load_session(PROC, s, XB, refs=(ref,))
            fn.band_power_session(recs[ref], fn.session_trials(K, s), os.path.join(BP_DIRS[ref], s + ".pkl"), verbose=False,
                                  meta_extra={"session": s, "ref": ref, "extra_bad_removed": removed})
            print(f"[{i}/{len(jobs)}] {s} {ref}: ok ({(time.time() - t0) / 60:.1f} min)", flush=True)
        except Exception as e:
            print(f"[{i}/{len(jobs)}] {s} {ref}: ERROR {e!r}", flush=True)
            pd.DataFrame([{"session": s, "ref": ref, "error": repr(e)}]).to_csv(os.path.join(TABLES, "band_power_errors.csv"), mode="a",
                         header=not os.path.exists(os.path.join(TABLES, "band_power_errors.csv")), index=False)
    print(f"band power finished in {(time.time() - t_all) / 60:.0f} min", flush=True)